# Dataset-to-Report LLM Pipeline

This module loads a dataset and delegates the computation of statistical or causal effects to a large language model. The model analyzes the data, performs the necessary mathematical reasoning, and generates a structured report summarizing the estimated effects and their interpretation.

In [1]:
import json
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, BadRequestError
import reportlab
import sys
from pathlib import Path
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))


from src.llm import generate_report_from_file_id

In [2]:
data_path = REPO_ROOT / "data" / "processed" / "berkeley_filtered.txt"
uploaded = client.files.create(
    file=open(data_path, "rb"),
    purpose="assistants",
)

text, latex = generate_report_from_file_id(uploaded.id, client)



USER_PROMPT: 
You are given a dataset as a PDF rendering of the dataset (table)."

Column roles:
- X (protected variable): <Sex>, with x0= Female and x1=Male
- Y (outcome attribute): <Admission> y= 'Accepted'
- Z (spurious features): <None>
- W (mediator variables): <Major>


Your task:
1. Analyze the dataset.
2. Compute the fairness decomposition according to Bareimboim and Plecko theory, both general and X-Z specific effects.
3. Produce a structured report.

Output format MUST be:

TEXT:
<plain language report>

LATEX:
<standalone LaTeX document>

USAGE: ResponseUsage(input_tokens=7520, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=6024, output_tokens_details=OutputTokensDetails(reasoning_tokens=4032), total_tokens=13544)


In [3]:
print(latex)

\documentclass{article}
\usepackage{geometry}
\geometry{margin=1in}
\usepackage{amsmath}
\begin{document}
Title: "Fairness Decomposition Report"

\subsection*{Overview of the Fairness Analysis}
This analysis decomposes the admission disparity between Sex groups in the provided dataset. We compare $X=\text{Female}$ (x0) versus $X=\text{Male}$ (x1). The outcome $Y$ is Admission with value "Accepted". We decomposed the observed difference in acceptance into total variation, the total causal effect, the indirect effect via the mediator Major, the direct effect (remaining causal effect not through Major), and the spurious effect; no confounders $Z$ are present in the data.

\subsection*{Decomposition of Effects}
\begin{itemize}
  \item \textbf{Total variation (observed difference in acceptance rate):}
    \begin{itemize}
      \item Acceptance probability for Female (x0): $0.4137$.
      \item Acceptance probability for Male (x1): $0.6459$.
      \item Total variation (Male $-$ Female): $0.

In [4]:
text

'Title: "Fairness Decomposition Report"\n\nOverview of the Fairness Analysis\nThis analysis decomposes the admission disparity between Sex groups in the provided dataset. We compare $X = $ Female (x0) versus $X = $ Male (x1). The outcome $Y$ is Admission with value "Accepted" (accepted vs. rejected). We decomposed the observed difference in acceptance into total variation, the total causal effect, the indirect effect via the mediator Major, the direct effect (remaining causal effect not through Major), and the spurious effect (confounding) — note there are no confounders in this dataset.\n\nDecomposition of Effects\n- Total variation (observed difference in acceptance rate):  \n  - Acceptance probability for Female (x0): 0.4137.  \n  - Acceptance probability for Male (x1): 0.6459.  \n  - Total variation (Male − Female): 0.2322.  \n  Interpretation: Observationally, Males are accepted about 23.22 percentage points more than Females.\n\n- Total causal effect $E[Y\\mid do(X=\\text{Male})]

In [5]:
latex

'\\documentclass{article}\n\\usepackage{geometry}\n\\geometry{margin=1in}\n\\usepackage{amsmath}\n\\begin{document}\nTitle: "Fairness Decomposition Report"\n\n\\subsection*{Overview of the Fairness Analysis}\nThis analysis decomposes the admission disparity between Sex groups in the provided dataset. We compare $X=\\text{Female}$ (x0) versus $X=\\text{Male}$ (x1). The outcome $Y$ is Admission with value "Accepted". We decomposed the observed difference in acceptance into total variation, the total causal effect, the indirect effect via the mediator Major, the direct effect (remaining causal effect not through Major), and the spurious effect; no confounders $Z$ are present in the data.\n\n\\subsection*{Decomposition of Effects}\n\\begin{itemize}\n  \\item \\textbf{Total variation (observed difference in acceptance rate):}\n    \\begin{itemize}\n      \\item Acceptance probability for Female (x0): $0.4137$.\n      \\item Acceptance probability for Male (x1): $0.6459$.\n      \\item Total